# TP4 : Factorisation en matrices non-négatives (NMF)

Importez les bibliothèques suivantes.

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

Les TPs précédents portaient sur le K-clustering et la PCA. Le présent TP utilisant les résultats de ces dernier, voici une cellule important les classes `KMeans` (**[documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)**) et `PCA` (**[documentation](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)**) de la bibliothèque `sklearn`. Regardez bien les documentations. Le nom des attributs et méthodes n'est pas nécessairement le même que dans les classes que nous avons nous même définies.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

def accuracy(labels):
    ct = pd.crosstab(labels, y)
    row_maxes = ct.max(axis=1)
    sum_of_maxes = row_maxes.sum()
    total_elements = ct.values.sum()
    proportion = sum_of_maxes / total_elements
    return proportion

Nous avons également besoin de la fonction affichant une image ou une liste d'images (au format 784 pixels).

In [ ]:
def display_image(source):
    if len(source) == 784:
        images = [source]
    elif len(source[0]) == 784:
        images = source
    else:
        print("error")

    l = len(images)
    c = math.ceil(l / 3)
    fig, axes = plt.subplots(c, 3, figsize=(12, 5))

    for i, ax in enumerate(axes.flat):
        if i < l:
            img = images[i].reshape(28, 28)
            ax.imshow(img, cmap='gray')
            ax.set_title(f"Image {i + 1}")
        ax.axis('off')

Enfin, importez les jeux de données.

In [ ]:
## from google.colab import files
## data_to_load1 = files.upload()
## import io
## df_pixels = pd.read_csv(io.BytesIO(data_to_load1['pixels.csv']))

df_pixels = pd.read_csv("pixels.csv")
X = df_pixels.to_numpy(dtype="float32")
df_label = pd.read_csv("labels.csv")
y = df_label["label"].values

FileNotFoundError: [Errno 2] No such file or directory: 'pixels.csv'

L'objectif de ce TP va consister en la définition progressive (par Monkey-Patching) d'une classe `MyNMF` implémentant l'algorithme de NMF de Lee and Seung.

In [ ]:
class MyNMF:
    def __init__(self, n_components, max_iter=500, tol=0.0005):
        self.n_components = n_components
        self.max_iter = max_iter
        self.tol = tol

        self.components = None
        self.reconstruction_err = []

print("Classe initialisée.")

Classe initialisée.


## Exercice 1

Construisez une fonction `compute_loss(data,W,H)` renvoyant la distance entre matrice `data` et $W \cdot H$ (à l'aide de la norme de Frobenius).

In [ ]:
def compute_loss(data,W,H):
    return np.linalg.norm(data - np.dot(W,H), 'fro')

## Exercice 2

Pour commencer l'implémentation de la classe, construisez une fonction `initialize_matrices(data,n_components)` renvoyant deux matrices `W,H` aux correctes dimensions initialisant l'algorithme de Lee et Seung avec des nombres positifs choisis aléatoirement.

In [ ]:
def initialize_matrices(data, n_components):
    n, m = data.shape
    Xc = np.mean(data, axis=1)
    W = np.random.rand(n, n_components) * np.reshape(Xc, (-1, 1))
    H = np.random.rand(n_components, m)
    return W, H



## Exercice 3

Nous allons maintenant définir une fonction `update_matrices(data, W, H, epsilon=1e-9)` renvoyant deux matrices `W,H` après une itération de l'algorithme de Lee et Seung. Le gros du travail va être de traduire la définition par élément de l'algorithme en termes matriciels (PAS DE BOUCLE FOR). Le terme `epsilon` a vocation a être ajouté aux dénominateurs afin d'éviter la division par 0.

In [ ]:



def update_matrices(data, W, H, epsilon=1e-9):

    WH = np.dot(W, H)


    ratio = data / (WH + epsilon)
    W = W * np.dot(ratio, H.T)

    # normalisation de W
    W = W / (np.sum(W, axis=0) + epsilon)


    WH = np.dot(W, H)
    ratio = data / (WH + epsilon)
    H = H * np.dot(W.T, ratio)

    return W, H


## Exercice 4

Nous allons maintenant construire la classe par Monkey-Patching.

1. Construisez une fonction `fit_transform(self, data)` implémentant la totalité de l'algorithme de Lee et Seung à l'aide des fonctions précedemments définies. Cette fonction renvoie `W` après optimisation et stocke `H` dans `self.components`. À chaque étape, on calcule l'erreur et on l'ajoute à la liste `self.reconstruction_err`. L'algorithme s'arrête lorsque le taux de variation de l'erreur passe sous le seuil de tolérance ou lorsqu'on atteint le nombre maximal d'itérations. Assignez la fonction à la classe `MyNMF`.

In [ ]:
def fit_transform(self, data):
    W, self.components = initialize_matrices(data, self.n_components)

    for i in range(self.max_iter):
        W, self.components = uptdate_matrices(data, W, self.components)

        loss = compute_loss(data,W,self.components)
        self.reconstruction_err.append(loss)

        if i > 2 and abs((self.reconstruction_err[-2] - self.reconstruction_err[-1])/ self.reconstruction_err[-2]) < self.tol:
            print(f"Convergence atteinte à l'itération {i}")
            break

    return W

MyNMF.fit_transform = fit_transform

2. Construisez une fonction `transform(self, data)` projettant une matrice `data` dans l'espace de la NMF. Construisez également une fonction `inverse_transform(self, data)` reconstruisant une matrice de données de l'espace de la NMF dans l'espace initial. Assignez lesfonctions à la classe `MyNMF`.

In [ ]:
def transform(self, data):
    n = data.shape[0]
    W = np.random.rand(n, self.n_components)

    for i in range(self.max_iter):
        num_W = np.dot(data, self.components.T)
        den_W = np.dot(W, np.dot(self.components, self.components.T)) + 1e-9
        W = W * (num_W / den_W)

    return W

def inverse_transform(self, data):
    return np.dot(data, self.components)

MyNMF.transform = transform
MyNMF.inverse_transform = inverse_transform

## Exercice 5

Il est maintenant temps d'appliquer notre classe à nos données.

1. Créez une instance de la classe `MyNMF` afin de faire une NMF sur nos données (`X`) avec 6 composantes. Affichez les images correspondants aux 6 composantes.

In [ ]:
my_nmf = MyNMF(n_components=6)
C = my_nmf.fit_transform(X)

display_image(my_nmf.components)

2. Choisissez une image dans nos données. Affichez là avec ses compressions (projection puis reconstruction) après une NMF avec $i$ composantes, pour $i$ prenant les valeurs 3, 5, 10, 50 et 100.

In [ ]:
indice = 245
img = [X[indice]]
for i in [3,5,10,50,100]:
    nmf = MyNMF(n_components=i)
    V = nmf.fit_transform(X)
    comp_image = nmf.inverse_transform(V)[indice]
    img.append(comp_image)
display_image(img)

2. Faites la même chose avec une PCA.

In [ ]:
indice = 245
img = [X[indice]]
for i in [3,5,10,50,100]:
    pca = PCA(n_components=i)
    V = pca.fit_transform(X)
    comp_image = pca.inverse_transform(V)[indice]
    img.append(comp_image)
display_image(img)

## Exercice 6

1. Utilisez la classe `MyNMF` pour projeter `X` sur deux dimensions. Visualisez le résultat dans un nuage de points avec `plt.scatter` (**[documentation](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.scatter.html#matplotlib.pyplot.scatter)**).

In [ ]:
nmf = MyNMF(n_components=2)
V = nmf.fit_transform(X)
X_projete = V
plt.scatter(X_projete[:,0],X_projete[:,1])
plt.show()

2. Dans ce nuage de points, colorez chaque point en fonction de son véritable label (le vecteur de labels est `y`).

In [ ]:
plt.scatter(X_projete[:,0],X_projete[:,1],c = y)
plt.show()

3. Changez la couleur des points pour qu'ils soient colorés en fonction de leur classification par un K-clustering de `X` avec 3 clusters. Affichez les images correspondants aux centroïdes. Affichez également la précision des labels obtenus.

In [ ]:
km = KMeans(n_clusters=3)
km.fit(X)
plt.scatter(X_projete[:,0],X_projete[:,1],c = km.labels_)
plt.show()
display_image(km.cluster_centers_)
print(accuracy(km.labels_))

4. Enfin, changez la couleur des points pour qu'ils soient colorés en fonction de leur classification par un K-clustering de la projection NMF de `X` en 2 dimensions, avec 3 clusters. Affichez les images correspondants aux centroïdes ainsi que la précision des labels obtenus.

In [ ]:
km = KMeans(n_clusters=3)
km.fit(X_projete)
plt.scatter(X_projete[:,0],X_projete[:,1],c = km.labels_)
plt.show()
display_image(nmf.inverse_transform(km.cluster_centers_))
print(accuracy(km.labels_))

## Exercice 7

1. Dessinez un graphe calculant la précision (`accuracy`) des labels obtenus par un K-clusterings de la projection NMF de `X` avec $i$ composantes avec 3 clusters, pour $i$ allant de 5 à 40 avec un pas de 5. Utilisez `plt.plot`. Faites le plusieurs fois, pour voir les différents minima locaux obtenus par optimisation.

In [ ]:
components = range(5,41,5)
accuracies = []
for n in components:
    nmf = MyNMF(n_components = n)
    X_proj = nmf.fit_transform(X)
    km = KMeans(n_clusters = 3)
    km.fit(X_proj)
    accuracies.append(accuracy(km.labels_))

plt.plot(components,accuracies)
plt.show()

2. Faîtes la même chose mais avec 20 clusters.

In [ ]:
components = range(5,41,5)
accuracies = []
for n in components:
    nmf = MyNMF(n_components = n)
    X_projete = nmf.fit_transform(X)
    my = KMeans(n_clusters = 20)
    my.fit(X_projete)
    accuracies.append(accuracy(my.labels_))

plt.plot(components,accuracies)
plt.show()

3. Comment interprétez-vous les résultats ?

Les résultats montrent que la précision du clustering change selon le nombre de composantes choisies pour la NMF. On remarque qu’elle est la plus élevée autour de 10 composantes, puis qu’elle diminue lorsque ce nombre augmente. Cela signifie qu’un nombre intermédiaire de composantes permet de mieux représenter les données.

Quand on utilise trop peu de composantes, une partie de l’information est perdue. À l’inverse, lorsqu’on en utilise trop, on ajoute du bruit, ce qui dégrade les performances.

# Exercice 8

À l'aune de tous ces résultats, quelles différences faites vous entre la NMF et la PCA dans le cadre de notre analyse de données ?

À partir des résultats obtenus, on remarque que la PCA et la NMF jouent des rôles différents dans l’analyse des données. La PCA est principalement utilisée pour réduire la dimension tout en conservant l’information globale. Par exemple, dans l’exercice 5, on observe que les images reconstruites avec la PCA deviennent progressivement de meilleure qualité lorsque le nombre de composantes augmente, même si elles restent assez floues.

La NMF, en revanche, impose uniquement des valeurs positives, ce qui conduit à des composantes plus simples à interpréter. Elle permet de mieux faire ressortir les structures présentes dans les données et se montre plus efficace pour le clustering.

En résumé, la PCA est surtout adaptée à la compression et à la visualisation des données, tandis que la NMF est plus pertinente pour l’interprétation et la création de groupes.